# 02 — VAYU Forecaster
> **Input:** `data/cleaned/00_shared/master_cleaned.csv`  
> **Outputs:** `models/xgb_6h.pkl`, `models/xgb_12h.pkl`, `models/xgb_24h.pkl`  
> **Run after:** `01_eda_and_cleaning.ipynb`

---

## What This Notebook Does

VAYU forecasts AQI (Air Quality Index) at **three future horizons** — +6 h, +12 h, and +24 h — using XGBoost regressors trained on the cleaned India AQI dataset (846 K hourly readings, 29 cities, 2022–2025).

| Horizon | Model file | Use case |
|---------|-----------|----------|
| +6 h | `xgb_6h.pkl` | Same-day planning, real-time alerts |
| +12 h | `xgb_12h.pkl` | Afternoon/evening outlook |
| +24 h | `xgb_24h.pkl` | Next-day advisory |

**Baseline to beat:** Linear Regression → R² = 0.585, RMSE = 45.14  
**Expected XGBoost performance:** R² 0.92 – 0.97 (per literature on Indian AQI data)

---

## Pipeline at a Glance

```
master_cleaned.csv
    │
    ├─ STEP 1  Load & audit
    ├─ STEP 2  Assign AQI column
    ├─ STEP 3  Sort chronologically per city
    ├─ STEP 4  Lag features  (AQI at t-1h, t-6h, t-24h)
    ├─ STEP 5  Future targets (AQI at t+6h, t+12h, t+24h)
    ├─ STEP 6  Drop non-model columns
    ├─ STEP 7  Select FINAL_FEATURES (14 columns)
    ├─ STEP 8  Drop NaNs introduced by lag/shift
    ├─ STEP 9  Define X and y arrays
    ├─ STEP 10 Chronological 80/20 train-test split
    ├─ STEP 11 Train Random Forest (baseline comparison)
    ├─ STEP 12 Train XGBoost (3 models × 3 horizons)
    ├─ STEP 13 Evaluate all models
    └─ STEP 14 Save XGBoost models + artefacts
```

> ⚠️ **Time-series rule:** the train/test split is **chronological** (no shuffle). Shuffling would leak future AQI values into training, inflating all metrics.


---
## Step 1 — Load the Dataset

We read `master_cleaned.csv` — the output of `01_eda_and_cleaning.ipynb`. This is the single source of truth for all downstream notebooks.

**Key columns used here:**
- `us_aqi` — the continuous AQI value we will forecast
- `city` — used to group lag features (prevents city boundary bleed)
- `datetime` — used for chronological sorting
- Six pollutant columns: `pm2_5_ugm3`, `pm10_ugm3`, `co_ugm3`, `no2_ugm3`, `so2_ugm3`, `o3_ugm3`


In [5]:
import pandas as pd

from pathlib import Path

ROOT_DIR = Path.cwd().parent

DATA_DIR = ROOT_DIR / "data"
MODELS_DIR = ROOT_DIR / "models"

MODELS_DIR.mkdir(exist_ok=True)

df = pd.read_csv(DATA_DIR / "cleaned/00_shared/master_cleaned.csv")

In [6]:
print(df.columns)

Index(['city', 'datetime', 'month', 'is_weekend', 'pm2_5_ugm3', 'pm10_ugm3',
       'co_ugm3', 'no2_ugm3', 'so2_ugm3', 'o3_ugm3', 'aqi_category', 'hour',
       'day_of_week', 'AQI', 'aqi_category_enc', 'city_enc'],
      dtype='str')


---
## Step 2 — Set the AQI Target Column

`us_aqi` is the continuous index computed from pollutant sub-indices using the CPCB piecewise-linear formula. We alias it to `AQI` for cleaner downstream references.


In [7]:
# df['AQI'] = df['us_aqi']

---
## Step 3 — Sort Chronologically by City

Sorting by `(city, datetime)` is **mandatory** before creating lag and lead features. Without it, lag features would draw values from a different city or a non-adjacent timestep, leaking information across city boundaries.


In [8]:
df = df.sort_values(['city', 'datetime'])

In [9]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['city_enc'] = le.fit_transform(df['city'])

In [10]:
import joblib
joblib.dump(le, MODELS_DIR / "city_encoder.pkl")

['/Users/rachitgoyal/Desktop/vayu-sul/models/city_encoder.pkl']

---
## Step 4 — Create Lag Features

Lag features give the model access to the recent pollution history at prediction time.

| Feature | Description |
|---------|-------------|
| `AQI_lag_1` | AQI one hour ago |
| `AQI_lag_6` | AQI six hours ago |
| `AQI_lag_24` | AQI 24 hours ago (same time yesterday) |

> **Critical:** lags are computed **within each city group** (`groupby('city')`). A naive `df['AQI'].shift(1)` would borrow the last reading from the previous city after the sort — silently corrupting training data.

These are the strongest predictive features in the final model (confirmed by SHAP analysis in `04_shap_explainer.ipynb`).


In [11]:
for lag in [1, 6, 24]:
    df[f'AQI_lag_{lag}'] = df.groupby('city')['AQI'].shift(lag)

---
## Step 5 — Create Future Targets

We use **negative shifts** to look ahead in time. For each row (city, hour), we record what the AQI will be 6, 12, and 24 hours later.

| Target column | Meaning |
|--------------|---------|
| `AQI_next_6` | AQI at t+6 h |
| `AQI_next_12` | AQI at t+12 h |
| `AQI_next_24` | AQI at t+24 h |

Rows at the tail of each city's time series will have NaN targets (no future observation available) — these are dropped in Step 8.


In [12]:
df['AQI_next_6']  = df.groupby('city')['AQI'].shift(-6)
df['AQI_next_12'] = df.groupby('city')['AQI'].shift(-12)
df['AQI_next_24'] = df.groupby('city')['AQI'].shift(-24)

In [13]:
df[['city','datetime','AQI','AQI_lag_1','AQI_lag_6','AQI_lag_24']].head(40)

,city,datetime,AQI,AQI_lag_1,AQI_lag_6,AQI_lag_24
0,agartala,2022-08-05 00:00:00,32.000000,NaN,NaN,NaN
1,agartala,2022-08-05 01:00:00,33.000000,32.000000,NaN,NaN
2,agartala,2022-08-05 02:00:00,34.000000,33.000000,NaN,NaN
3,agartala,2022-08-05 03:00:00,32.000000,34.000000,NaN,NaN
4,agartala,2022-08-05 04:00:00,30.333333,32.000000,NaN,NaN
5,agartala,2022-08-05 05:00:00,31.833333,30.333333,NaN,NaN
6,agartala,2022-08-05 06:00:00,31.000000,31.833333,32.000000,NaN
7,agartala,2022-08-05 07:00:00,39.000000,31.000000,33.000000,NaN
8,agartala,2022-08-05 08:00:00,50.000000,39.000000,34.000000,NaN
9,agartala,2022-08-05 09:00:00,60.000000,50.000000,32.000000,NaN


---
## Step 6 — Drop Non-Model Columns

We remove columns that are either redundant, leaking, or not available at inference time:
- **Identifiers:** `city`, `state`, `datetime` (city identity is captured by `city_enc`)
- **Geographic:** `latitude`, `longitude`
- **Derived categoricals:** `day_name`, `season`, `time_of_day`
- **Environmental extras:** `dust_ugm3`, `aod` (Aerosol Optical Depth — not available from standard AQI stations)
- **Binary flags:** `festival_period`, `crop_burning_season`, `is_raining`, `heavy_rain`
- **Label columns:** `aqi_category`, `pm25_category_india`

After dropping, the dataframe contains only numerical features and targets.


In [14]:
df = df.drop(columns=[
    'city','state','latitude','longitude','datetime',
    'day_name','season','time_of_day',
    'dust_ugm3','aod',
    'aqi_category','pm25_category_india',
    'festival_period','crop_burning_season',
    'is_raining','heavy_rain'
], errors='ignore')

In [15]:
df['is_weekend'] = df['is_weekend'].astype(str).map({
    'True': 1,
    'False': 0
}).fillna(0).astype(int)

---
## Step 7 — Define FINAL_FEATURES

Feature selection was performed in `01_eda_and_cleaning.ipynb` via:
1. Correlation filter (dropped features with inter-feature correlation > 0.85)
2. Random Forest importance filter (dropped features with importance < 0.02)

The 14 retained features are:

| Group | Features |
|-------|---------|
| Pollutants | `pm2_5_ugm3`, `pm10_ugm3`, `co_ugm3`, `no2_ugm3`, `so2_ugm3`, `o3_ugm3` |
| Time | `hour`, `day_of_week`, `month`, `is_weekend` |
| Location | `city_enc` |
| History | `AQI_lag_1`, `AQI_lag_6`, `AQI_lag_24` |

The `FEATURES` list is also saved to `models/features.pkl` so the inference API can load it without hardcoding.


In [16]:
FEATURES = [
    # pollutants
    'pm2_5_ugm3','pm10_ugm3','co_ugm3','no2_ugm3','so2_ugm3','o3_ugm3',

    # time
    'hour','day_of_week','month','is_weekend',

    # location
    'city_enc',

    # lag features
    'AQI_lag_1','AQI_lag_6','AQI_lag_24'
]

joblib.dump(FEATURES, MODELS_DIR / "features.pkl")

['/Users/rachitgoyal/Desktop/vayu-sul/models/features.pkl']

---
## Step 8 — Drop NaN Rows

Shifting creates NaN values at the edges of each city's time series (no past/future to reference). We drop all rows where any feature or target is NaN.

Expect to lose roughly `24 rows × 29 cities = ~700 rows` — a negligible fraction of 829 K rows.


In [17]:
df = df.dropna()

---
## Step 9 — Define X and y Arrays

We create three separate target arrays — one per forecast horizon. All three share the same feature matrix `X`.


In [18]:
X = df[FEATURES]

y_6  = df['AQI_next_6']
y_12 = df['AQI_next_12']
y_24 = df['AQI_next_24']

---
## Step 10 — Chronological Train / Test Split (80 / 20)

```
──────────────────────────────────────────────────────────
  TRAIN (80%)                          │  TEST (20%)
  2022-01 ─────────────────── 2024-07  │  2024-08 ─── 2025
──────────────────────────────────────────────────────────
```

We sort the dataframe by `datetime` and take the **first 80%** of rows as training data. The **last 20%** is the test set.

> **Why not shuffle?** AQI is a time series. If we shuffled, the model would train on 2025 data to predict 2023 — an impossible scenario at deployment. Chronological splitting ensures the evaluation reflects real-world inference conditions.


In [19]:
split = int(len(df) * 0.8)

X_train, X_test = X.iloc[:split], X.iloc[split:]
y6_train, y6_test = y_6.iloc[:split], y_6.iloc[split:]
y12_train, y12_test = y_12.iloc[:split], y_12.iloc[split:]
y24_train, y24_test = y_24.iloc[:split], y_24.iloc[split:]

---
## Step 11a — Train Random Forest Regressors (Baseline)

Random Forest is trained here as a **comparison baseline**. Three models are trained — one per horizon — using `n_estimators=200` and `n_jobs=-1` for full parallelism.

Random Forest is **not** the deployment model because at ~2–3 GB per pkl file it cannot be served on Render's free tier. XGBoost models are ~5 MB each.

The RF results are included in the final comparison table so the performance delta is documented.


In [20]:
from sklearn.ensemble import RandomForestRegressor

rf_models = {}

for y_train, label in [
    (y6_train, "6h"),
    (y12_train, "12h"),
    (y24_train, "24h")
]:
    print(f"\nTraining RF for {label}")
    
    rf = RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
        verbose=1
    )
    
    rf.fit(X_train, y_train)
    
    rf_models[label] = rf


Training RF for 6h


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=-1)]: Done  30 tasks      | elapsed:   23.7s
[Parallel(n_jobs=-1)]: Done 180 tasks      | elapsed:  2.2min
[Parallel(n_jobs=-1)]: Done 200 out of 200 | elapsed:  2.5min finished
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 10 concurrent workers.



Training RF for 12h


[Parallel(n_jobs=-1)]: Done  30 tasks      | elapsed:   21.3s
[Parallel(n_jobs=-1)]: Done 180 tasks      | elapsed:  2.2min
[Parallel(n_jobs=-1)]: Done 200 out of 200 | elapsed:  2.5min finished
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 10 concurrent workers.



Training RF for 24h


[Parallel(n_jobs=-1)]: Done  30 tasks      | elapsed:   23.3s
[Parallel(n_jobs=-1)]: Done 180 tasks      | elapsed:  2.3min
[Parallel(n_jobs=-1)]: Done 200 out of 200 | elapsed:  2.6min finished


---
## Step 11b — Evaluate Random Forest

We define a shared `evaluate_model` function that computes R² and RMSE. The same function is reused for XGBoost evaluations.

| Metric | Meaning |
|--------|---------|
| **R²** | Proportion of AQI variance explained (1.0 = perfect) |
| **RMSE** | Root-mean-squared error in AQI units (lower = better) |


In [21]:
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error

def evaluate_model(model, X_test, y_test, name):
    preds = model.predict(X_test)
    r2 = r2_score(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    
    print(f"{name}")
    print("R2:", r2)
    print("RMSE:", rmse)
    print()

---
## Step 12 — Train XGBoost Regressors (Deployment Models)

Three separate `XGBRegressor` instances are trained — one per forecast horizon. XGBoost is the deployment choice because:

- **Model size:** ~5 MB vs ~2–3 GB for Random Forest — fits comfortably on Render free tier
- **Speed:** inference in microseconds per request
- **Accuracy:** R² 0.92–0.97 expected on Indian AQI data (per literature)

**Hyperparameters:**

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| `n_estimators` | 300 | More trees than RF because boosting benefits from iterations |
| `learning_rate` | 0.05 | Conservative — reduces overfitting risk |
| `max_depth` | — | Default (6); moderate regularisation |
| `tree_method` | `hist` | Fast histogram-based splits; required for large datasets |
| `random_state` | 42 | Reproducibility |


In [22]:
pd.DataFrame({
    "Actual": y6_test.values[:20],
    "Predicted": model_6.predict(X_test)[:20]
})

NameError: name 'model_6' is not defined

In [ ]:
!pip install xgboost


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
from xgboost import XGBRegressor

model_6  = XGBRegressor(n_estimators=300, learning_rate=0.05, random_state=42)
model_12 = XGBRegressor(n_estimators=300, learning_rate=0.05, random_state=42)
model_24 = XGBRegressor(n_estimators=300, learning_rate=0.05, random_state=42)

model_6.fit(X_train, y6_train)
model_12.fit(X_train, y12_train)
model_24.fit(X_train, y24_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes

---
## Step 13 — Evaluate XGBoost Models

We evaluate each horizon model against the held-out test set. Results are compared against the linear regression baseline (R² = 0.585, RMSE = 45.14) and the Random Forest baseline from Step 11.

**What to look for:**
- R² should be > 0.90 — if not, check for lag feature bleed across city boundaries
- RMSE should be < 20 AQI units for the +6h model; slightly higher for +12h and +24h (uncertainty grows with horizon)
- All three horizons should outperform linear regression by a large margin


In [ ]:
!pip install -U scikit-learn

c:\Users\Hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedWriter name=3>
  res = process_handler(cmd, _system_body)
c:\Users\Hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedReader name=4>
  res = process_handler(cmd, _system_body)
c:\Users\Hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedReader name=5>
  res = process_handler(cmd, _system_body)


In [ ]:
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

def evaluate(y_true, y_pred, name):
    print(f"==== {name} ====")
    print("R2   :", r2_score(y_true, y_pred))
    
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    
    print("RMSE :", rmse)
    print()

In [ ]:
evaluate(y6_test, model_6.predict(X_test), "6h")
evaluate(y12_test, model_12.predict(X_test), "12h")
evaluate(y24_test, model_24.predict(X_test), "24h")

==== 6h ====
R2   : 0.4802596958666402
RMSE : 37.22535693276344

==== 12h ====
R2   : 0.4273528027809399
RMSE : 39.061306482028314

==== 24h ====
R2   : 0.5516458013672116
RMSE : 34.58522671302168



---
## Step 13b — Sanity Check: Actual vs Predicted

A quick visual check of the first 20 test predictions against ground truth. Large systematic errors here indicate a data preparation bug (e.g. lag bleed, wrong column order) rather than a model tuning issue.


In [ ]:
pd.DataFrame({
    "Actual": y6_test.values[:20],
    "Predicted": model_6.predict(X_test)[:20]
})

,Actual,Predicted
0,117.044828,110.502693
1,107.486207,106.182213
2,97.738462,98.030228
3,94.000000,100.685768
4,129.074627,116.205452
5,166.014925,152.115143
6,171.925373,163.142685
7,167.492537,187.437378
8,154.194030,204.833588
9,146.805970,195.876801


In [ ]:
joblib.dump(model_6,  MODELS_DIR / "xgb_6h.pkl")
joblib.dump(model_12, MODELS_DIR / "xgb_12h.pkl")
joblib.dump(model_24, MODELS_DIR / "xgb_24h.pkl")

['models/city_encoder.pkl']

---
## Step 14a — Save Models & Artefacts

We persist three XGBoost models and the shared feature list:

| File | Contents |
|------|----------|
| `models/xgb_6h.pkl` | +6h forecast model |
| `models/xgb_12h.pkl` | +12h forecast model |
| `models/xgb_24h.pkl` | +24h forecast model |
| `models/features.pkl` | Ordered feature list (14 columns) |
| `models/city_encoder.pkl` | LabelEncoder for city → integer |

The FastAPI backend (`backend/predict.py`) loads these at startup and serves predictions via REST endpoints.


In [ ]:
m6 = joblib.load(MODELS_DIR / "xgb_6h.pkl")

sample = X_test.iloc[0].values.reshape(1, -1)
print(m6.predict(sample))

[110.50269]


In [ ]:
pd.DataFrame({
    "Actual": y6_test.values[:20],
    "Predicted": model_6.predict(X_test)[:20]
})

,Actual,Predicted
0,117.044828,110.502693
1,107.486207,106.182213
2,97.738462,98.030228
3,94.000000,100.685768
4,129.074627,116.205452
5,166.014925,152.115143
6,171.925373,163.142685
7,167.492537,187.437378
8,154.194030,204.833588
9,146.805970,195.876801


In [ ]:
print("===== RANDOM FOREST =====")
evaluate_model(rf_models["6h"], X_test, y6_test, "RF 6h")
evaluate_model(rf_models["12h"], X_test, y12_test, "RF 12h")
evaluate_model(rf_models["24h"], X_test, y24_test, "RF 24h")

===== RANDOM FOREST =====


[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    2.0s
[Parallel(n_jobs=16)]: Done 168 tasks      | elapsed:   17.7s
[Parallel(n_jobs=16)]: Done 200 out of 200 | elapsed:   25.6s finished


RF 6h
R2: 0.5856083215404093
RMSE: 33.23924765813945



[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    8.7s
[Parallel(n_jobs=16)]: Done 168 tasks      | elapsed:   23.1s
[Parallel(n_jobs=16)]: Done 200 out of 200 | elapsed:   24.7s finished


RF 12h
R2: 0.337631053169686
RMSE: 42.010047940517104



[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    2.3s
[Parallel(n_jobs=16)]: Done 168 tasks      | elapsed:   12.5s


RF 24h
R2: 0.6362253720678768
RMSE: 31.152737020614627



[Parallel(n_jobs=16)]: Done 200 out of 200 | elapsed:   16.3s finished


In [ ]:
print("===== XGBOOST =====")
evaluate_model(model_6, X_test, y6_test, "XGB 6h")
evaluate_model(model_12, X_test, y12_test, "XGB 12h")
evaluate_model(model_24, X_test, y24_test, "XGB 24h")

===== XGBOOST =====
XGB 6h
R2: 0.970813164862302
RMSE: 6.744911115104681

XGB 12h
R2: 0.9043127591234664
RMSE: 12.213651545519935

XGB 24h
R2: 0.7745511813432562
RMSE: 18.75464663062243



In [ ]:
import pandas as pd

results = []

def collect_results(model, X_test, y_test, name):
    preds = model.predict(X_test)
    results.append({
        "Model": name,
        "R2": r2_score(y_test, preds),
        "RMSE": np.sqrt(mean_squared_error(y_test, preds))
    })

# RF
collect_results(rf_models["6h"], X_test, y6_test, "RF 6h")
collect_results(rf_models["12h"], X_test, y12_test, "RF 12h")
collect_results(rf_models["24h"], X_test, y24_test, "RF 24h")

# XGB
collect_results(model_6, X_test, y6_test, "XGB 6h")
collect_results(model_12, X_test, y12_test, "XGB 12h")
collect_results(model_24, X_test, y24_test, "XGB 24h")

pd.DataFrame(results)

[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done  30 tasks      | elapsed:    0.5s
[Parallel(n_jobs=10)]: Done 180 tasks      | elapsed:    3.3s
[Parallel(n_jobs=10)]: Done 200 out of 200 | elapsed:    3.5s finished
[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done  30 tasks      | elapsed:    0.5s
[Parallel(n_jobs=10)]: Done 180 tasks      | elapsed:    2.6s
[Parallel(n_jobs=10)]: Done 200 out of 200 | elapsed:    2.9s finished
[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done  30 tasks      | elapsed:    0.6s
[Parallel(n_jobs=10)]: Done 180 tasks      | elapsed:    3.0s
[Parallel(n_jobs=10)]: Done 200 out of 200 | elapsed:    3.2s finished


,Model,R2,RMSE
0,RF 6h,0.970467,6.784769
1,RF 12h,0.890439,13.069153
2,RF 24h,0.819950,16.760277
3,XGB 6h,0.970813,6.744911
4,XGB 12h,0.904313,12.213652
5,XGB 24h,0.774551,18.754647


---
## Step 14b — Save Random Forest 24h Model

The 24h Random Forest model is saved separately for archival / offline analysis. It is **not** loaded by the production API.


In [ ]:
rf_24 = rf_models["24h"]
joblib.dump(rf_24, MODELS_DIR / "rf_24h.pkl")

['models/rf_24h.pkl']

---
## Summary — Forecaster Outputs

| File | Size (approx) | Used by |
|------|--------------|---------|
| `models/xgb_6h.pkl` | ~5 MB | FastAPI `/predict/6h` |
| `models/xgb_12h.pkl` | ~5 MB | FastAPI `/predict/12h` |
| `models/xgb_24h.pkl` | ~5 MB | FastAPI `/predict/24h` |
| `models/features.pkl` | < 1 KB | All inference scripts |
| `models/city_encoder.pkl` | < 1 KB | All inference scripts |

**Next step:** run `03_classifier.ipynb` to train the AQI category classifier.  
**Or:** run `04_shap_explainer.ipynb` to generate SHAP attribution profiles.
